In [2]:
import os
import json
import torch
import torch.nn.functional as F
import numpy as np
from torch.utils.data import DataLoader
from torchvision.models import inception_v3
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from scipy.stats import entropy
from diffusers import UNet2DConditionModel, AutoencoderKL, DDPMScheduler
from transformers import CLIPTextModel, CLIPTokenizer

In [3]:
# -----------------------------
# CONFIG
# -----------------------------
COCO_PATH = "/teamspace/studios/this_studio/coco2014"
CHECKPOINT_PATH = "/teamspace/studios/this_studio/Latent-Diffusion-Model-for-text-to-image-generation/ldm_checkpoints/epoch_4"
BATCH_SIZE = 128
NUM_SAMPLES = 2000   # Use 5k–10k for stable IS
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
LATENT_SIZE = 16
NUM_INFERENCE_STEPS = 250
GUIDANCE_SCALE = 7.5

In [4]:
# -----------------------------
# DATASET
# -----------------------------

class CocoValCaptions(torch.utils.data.Dataset):
    def __init__(self, path, tokenizer):
        with open(f'{path}/annotations/captions_val2014.json') as f:
            self.data = json.load(f)['annotations']
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        caption = self.data[idx]['caption']
        tokens = self.tokenizer(
            caption,
            padding="max_length",
            truncation=True,
            max_length=77,
            return_tensors="pt"
        )
        return tokens.input_ids.squeeze(0)



In [5]:
# -----------------------------
# LOAD MODELS ONCE
# -----------------------------

print("Loading models...")

unet = UNet2DConditionModel.from_pretrained(
    os.path.join(CHECKPOINT_PATH, "unet"),
    use_safetensors=True
).to(DEVICE)

vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(DEVICE)
text_encoder = CLIPTextModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
scheduler = DDPMScheduler(num_train_timesteps=1000, beta_schedule="linear")

checkpoint = torch.load(os.path.join(CHECKPOINT_PATH, "training_state.pth"), map_location=DEVICE)
unet.load_state_dict(checkpoint['model_state_dict'])

unet.eval()
vae.eval()
text_encoder.eval()

print("Models loaded.")


Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


Loading models...


Cannot initialize model with low cpu memory usage because `accelerate` was not found in the environment. Defaulting to `low_cpu_mem_usage=False`. It is strongly recommended to install `accelerate` for faster and less memory-intense model loading. You can do so with: 
```
pip install accelerate
```
.


Models loaded.


In [6]:
# -----------------------------
# LOAD INCEPTION
# -----------------------------

inception = inception_v3(pretrained=True, transform_input=False).to(DEVICE)
inception.eval()

resize = transforms.Resize((299, 299))


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=Inception_V3_Weights.IMAGENET1K_V1`. You can also use `weights=Inception_V3_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [7]:
# -----------------------------
# GENERATION FUNCTION (BATCHED)
# -----------------------------
@torch.no_grad()
def generate_batch(input_ids):
    batch_size = input_ids.shape[0]

    input_ids = input_ids.to(DEVICE)

    text_embeddings = text_encoder(input_ids)[0]

    uncond_input = tokenizer(
        [""] * batch_size,
        padding="max_length",
        max_length=77,
        return_tensors="pt"
    ).to(DEVICE)

    uncond_embeddings = text_encoder(uncond_input.input_ids)[0]

    text_embeddings = torch.cat([uncond_embeddings, text_embeddings])

    latents = torch.randn(
        (batch_size, 4, LATENT_SIZE, LATENT_SIZE),
        device=DEVICE
    )

    latents *= scheduler.init_noise_sigma
    scheduler.set_timesteps(NUM_INFERENCE_STEPS)

    for t in scheduler.timesteps:
        latent_model_input = torch.cat([latents] * 2)
        latent_model_input = scheduler.scale_model_input(latent_model_input, t)

        noise_pred = unet(
            latent_model_input,
            t,
            encoder_hidden_states=text_embeddings
        ).sample

        noise_pred_uncond, noise_pred_text = noise_pred.chunk(2)
        noise_pred = noise_pred_uncond + GUIDANCE_SCALE * (noise_pred_text - noise_pred_uncond)

        latents = scheduler.step(noise_pred, t, latents).prev_sample

    latents = 1 / 0.18215 * latents
    images = vae.decode(latents).sample

    images = (images / 2 + 0.5).clamp(0, 1)

    return images

In [8]:
# -----------------------------
# INCEPTION SCORE COMPUTATION
# -----------------------------

@torch.no_grad()
def compute_inception_score(preds, splits=10):
    N = preds.shape[0]
    split_scores = []

    for k in range(splits):
        part = preds[k * (N // splits):(k + 1) * (N // splits)]
        py = np.mean(part, axis=0)

        scores = []
        for i in range(part.shape[0]):
            scores.append(entropy(part[i], py))

        split_scores.append(np.exp(np.mean(scores)))

    return np.mean(split_scores), np.std(split_scores)

In [9]:
import json
import os
from collections import OrderedDict

SAVE_PATH = "inception_step_metrics.json"
torch.cuda.empty_cache()
# You can extend this list anytime
STEPS_LIST = [25, 50, 75, 100, 150, 200,250,300,350,400,450]

# -----------------------------
# Load existing metrics if present
# -----------------------------
if os.path.exists(SAVE_PATH):
    with open(SAVE_PATH, "r") as f:
        metrics_map = OrderedDict(json.load(f))
    print("Loaded existing metrics:")
    print(metrics_map)
else:
    metrics_map = OrderedDict()

# -----------------------------
# Automatically determine remaining steps
# -----------------------------
completed_steps = set(int(k) for k in metrics_map.keys())
remaining_steps = [s for s in STEPS_LIST if s not in completed_steps]

if not remaining_steps:
    print("\nAll steps already evaluated.")
else:
    print(f"\nRemaining steps to evaluate: {remaining_steps}")

# -----------------------------
# Dataset (fixed subset)
# -----------------------------
dataset = CocoValCaptions(COCO_PATH, tokenizer)
subset = torch.utils.data.Subset(dataset, range(NUM_SAMPLES))
loader = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=False)

print("\nStarting step-wise evaluation...\n")

# -----------------------------
# Main loop (only remaining steps)
# -----------------------------
for steps in remaining_steps:

    print(f"\nEvaluating for {steps} diffusion steps")

    NUM_INFERENCE_STEPS = steps
    scheduler.set_timesteps(NUM_INFERENCE_STEPS)

    all_preds = []
    generated = 0

    for input_ids in tqdm(loader):

        with torch.autocast("cuda"):
            images = generate_batch(input_ids)

        images = resize(images)
        images = images.float()

        logits = inception(images)
        probs = F.softmax(logits, dim=1)

        all_preds.append(probs.cpu().detach().numpy())

        generated += images.shape[0]
        if generated >= NUM_SAMPLES:
            break

    all_preds = np.concatenate(all_preds, axis=0)
    all_preds = all_preds[:NUM_SAMPLES]

    mean_is, std_is = compute_inception_score(all_preds)

    metrics_map[str(steps)] = {
        "mean_is": float(mean_is),
        "std_is": float(std_is)
    }

    print(f"IS @ {steps} steps: {mean_is:.4f} ± {std_is:.4f}")

    # -----------------------------
    # SAVE AFTER EACH STEP
    # -----------------------------
    with open(SAVE_PATH, "w") as f:
        json.dump(metrics_map, f, indent=4)

    print(f"Saved progress to {SAVE_PATH}")

print("\nFinal Step → Metric Mapping:")
print(metrics_map)

Loaded existing metrics:
OrderedDict({'25': {'mean_is': 6.241247653961182, 'std_is': 0.3918217122554779}, '50': {'mean_is': 6.495041847229004, 'std_is': 0.5195876359939575}, '75': {'mean_is': 6.642598628997803, 'std_is': 0.4450549781322479}, '100': {'mean_is': 6.404843330383301, 'std_is': 0.6089599132537842}, '150': {'mean_is': 6.509793758392334, 'std_is': 0.3705785572528839}, '200': {'mean_is': 6.443101406097412, 'std_is': 0.4162690043449402}, '250': {'mean_is': 6.364321708679199, 'std_is': 0.5629221796989441}})

Remaining steps to evaluate: [300, 350, 400, 450]

Starting step-wise evaluation...


Evaluating for 300 diffusion steps


 94%|█████████▍| 15/16 [04:43<00:18, 18.92s/it]


IS @ 300 steps: 6.3641 ± 0.4577
Saved progress to inception_step_metrics.json

Evaluating for 350 diffusion steps


 94%|█████████▍| 15/16 [05:16<00:21, 21.10s/it]


IS @ 350 steps: 6.4411 ± 0.3859
Saved progress to inception_step_metrics.json

Evaluating for 400 diffusion steps


 94%|█████████▍| 15/16 [05:54<00:23, 23.63s/it]


IS @ 400 steps: 6.2733 ± 0.3543
Saved progress to inception_step_metrics.json

Evaluating for 450 diffusion steps


 94%|█████████▍| 15/16 [06:40<00:26, 26.68s/it]

IS @ 450 steps: 6.2450 ± 0.5163
Saved progress to inception_step_metrics.json

Final Step → Metric Mapping:
OrderedDict({'25': {'mean_is': 6.241247653961182, 'std_is': 0.3918217122554779}, '50': {'mean_is': 6.495041847229004, 'std_is': 0.5195876359939575}, '75': {'mean_is': 6.642598628997803, 'std_is': 0.4450549781322479}, '100': {'mean_is': 6.404843330383301, 'std_is': 0.6089599132537842}, '150': {'mean_is': 6.509793758392334, 'std_is': 0.3705785572528839}, '200': {'mean_is': 6.443101406097412, 'std_is': 0.4162690043449402}, '250': {'mean_is': 6.364321708679199, 'std_is': 0.5629221796989441}, '300': {'mean_is': 6.3640923500061035, 'std_is': 0.4577328860759735}, '350': {'mean_is': 6.441117763519287, 'std_is': 0.38593316078186035}, '400': {'mean_is': 6.273316860198975, 'std_is': 0.3542514443397522}, '450': {'mean_is': 6.24500036239624, 'std_is': 0.5162811875343323}})
